# validation check — delta buckets (market − lrr) → blanket DRAW, early season

Split matches into `N_BUCKETS` (base 6) equal-count buckets of `delta = market − elo(lrr)`; boundaries
fit on the **train fold only**. Rule: bet the middle buckets (all but the first and last) on the DRAW. Blanket model. Population = early season (gw ≤ 8).
Rough-draft partition across three notebooks: first bucket → AWAY, middle → DRAW, last bucket → HOME.
`N_BUCKETS` is a knob — swept 3..14 at the bottom. Region/outcome fixed from the full-sample look →
meta-DoF; gates test time/league robustness.

In [1]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE — model space (ignored by the blanket model; kept for dropna) -----
MARKET = "mrkt_home_away_impl_diff"
ELO    = "teams_elo_diff_lrr_impl_diff"
FEATURES = [MARKET, ELO]

In [3]:
# --- TARGET ------------------------------------------------------------------
OUTCOME = "t_draw_flg"
IMPLIED = "mrkt_draw_impl"

In [4]:
# --- ABT FILTERING — early season (gw <= 8) ---------------------------------
MAX_GAMEWEEK = 8
matches = abt[abt["gameweek"] <= MAX_GAMEWEEK].dropna(
    subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()
print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

matches: 1544   seasons: ['2223', '2324', '2425', '2526']   leagues: ['england', 'france', 'germany', 'italy', 'spain']


In [5]:
# --- SEGMENTER — delta buckets (bet middle (all but first & last)) ----------------------------
# delta = market - elo(lrr). N_BUCKETS equal-count buckets, boundaries fit on TRAIN only. N_BUCKETS is
# a knob (swept below). Contract: segmenter(train) -> labeler ; labeler(rows) -> Series (NaN = skip).
N_BUCKETS = 6

def _delta(df):
    return df[MARKET] - df[ELO]

def region_segmenter(n_buckets):
    def segmenter(train):
        d = _delta(train)
        lo = d.quantile(1 / n_buckets)
        hi = d.quantile((n_buckets - 1) / n_buckets)
        def labeler(rows):
            dd = _delta(rows)
            lab = pd.Series(np.nan, index=rows.index, dtype="float64")
            lab[(dd >= lo) & (dd <= hi)] = 1.0
            return lab
        return labeler
    return segmenter

segmenter = region_segmenter(N_BUCKETS)

# Self-description (in-sample): the N_BUCKETS delta buckets; bet buckets marked.
_d = _delta(matches)
_b = pd.qcut(_d, N_BUCKETS, labels=False)
print(f"in-sample delta buckets (N_BUCKETS={N_BUCKETS}; bet middle (all but first & last) -> DRAW)")
print(" bucket      n   delta    win%  impl%   edge     ROI   bet")
for b in range(N_BUCKETS):
    m = matches[_b == b]
    wr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    bet = "DRAW" if 1 <= b <= N_BUCKETS - 2 else "  --"
    print(f" {b:>6} {len(m):>7}  {_d[_b == b].mean():+6.3f}   {wr*100:5.1f}  {im*100:5.1f}  "
          f"{(wr-im)*100:+5.1f}  {pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {bet}")

in-sample delta buckets (N_BUCKETS=6; bet middle (all but first & last) -> DRAW)
 bucket      n   delta    win%  impl%   edge     ROI   bet
      0     258  -0.159    23.3   26.4   -3.1  -0.119    --
      1     258  -0.066    29.8   27.0   +2.8  +0.104  DRAW
      2     258  -0.014    27.1   26.1   +1.0  +0.040  DRAW
      3     256  +0.030    30.5   25.8   +4.6  +0.179  DRAW
      4     257  +0.076    25.3   24.7   +0.6  +0.023  DRAW
      5     257  +0.164    21.4   25.0   -3.6  -0.145    --


In [6]:
# --- MODEL — blanket bettor -------------------------------------------------
class BlanketBettor:
    def fit(self, X, y):
        self.classes_ = np.array([0, 1])
        return self
    def predict_proba(self, X):
        n = len(X)
        return np.column_stack([np.zeros(n), np.ones(n)])

def make_model():
    return BlanketBettor()

BUFFER = 0.0

## Summary

In [7]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="elodelta_draw_middle_blanket", space="elo_delta", target="draw",
              abt_filter="major, gw<=8, dropna",
              segmentation=f"delta=market-lrr, {N_BUCKETS} qcut buckets (train-fit), bet middle (all but first & last)",
              segment_filter=f"middle of {N_BUCKETS} (blanket)")

elodelta_draw_middle_blanket   pooled +0.083 (1030 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           FAIL  (+2/4 seasons)
    2223 +0.102    2324 -0.031    2425 +0.272    2526 -0.012

GATE 2  league — ROI by league / drop best   PASS
    france +0.235    italy +0.217    england +0.074    germany +0.018    spain -0.106
    pooled without best league (italy): +0.045

GATE 3  walk-forward — ROI by test season    PASS
    2425 +0.278    2526 -0.012


## Extra checks — below the template

In [8]:
# EXTRA CHECK — knob robustness: number of delta buckets n = 3..14 (bet middle (all but first & last) -> DRAW).
# A real edge survives a range of n; one that appears at a single split is fragile.
print(" n_buckets   pooled    profit    bets   seasons+   bet_frac")
for n in range(3, 15):
    b = collect_bets(matches, season_folds(matches), features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
                     segmenter=region_segmenter(n), make_model=make_model, buffer=BUFFER)
    ps = roi_grouped(b, "fold")
    print(f" {n:<9} {pooled_roi(b['implied'], b['won']):+.4f}  {(b['won'] - b['implied']).sum():+8.2f}  "
          f"{len(b):>5}   +{int((ps > 0).sum())}/{len(ps)}      {(n - 2) / n:.2f}")

 n_buckets   pooled    profit    bets   seasons+   bet_frac


 3         +0.0876    +11.68    513   +4/4      0.33
 4         +0.0697    +13.88    769   +2/4      0.50


 5         +0.0989    +23.67    924   +2/4      0.60


 6         +0.0830    +22.14   1030   +2/4      0.67
 7         +0.0851    +24.23   1102   +2/4      0.71


 8         +0.0794    +23.63   1150   +3/4      0.75


 9         +0.0666    +20.72   1202   +3/4      0.78
 10        +0.0649    +20.78   1235   +3/4      0.80


 11        +0.0662    +21.61   1258   +3/4      0.82


 12        +0.0570    +18.98   1285   +3/4      0.83
 13        +0.0478    +16.24   1312   +2/4      0.85


 14        +0.0491    +16.87   1325   +4/4      0.86
